# Lab 02 — Cleaning & Normalization Pipeline

Цей ноутбук очікує, що `project_lab1` знаходиться поруч із `project_lab2` у спільній папці `nlp/`.
Він читає `../project_lab1/data/raw.jsonl` + `../project_lab1/data/labels.csv`, генерує `data/processed_v2/processed_v2.csv` та `docs/audit_summary_lab2.md`, а також оновлює `docs/preprocess_policy.md` прикладами до/після.

## 1) Install deps (Colab / local)

In [1]:
!pip -q install -r ../requirements.txt

## 2) Paths + import pipeline

In [2]:
from pathlib import Path
import json
import pandas as pd
from tqdm import tqdm
import sys

LAB2_ROOT = Path('..').resolve()
LAB1_ROOT = (LAB2_ROOT.parent / 'project_lab1').resolve()

print('LAB2_ROOT:', LAB2_ROOT)
print('LAB1_ROOT:', LAB1_ROOT)

sys.path.insert(0, str(LAB2_ROOT))

from src.preprocess import preprocess, idempotence_check, no_empty_explosion

LAB2_ROOT: C:\Users\maia1\data\politiekh\masters\nlp\project_lab2
LAB1_ROOT: C:\Users\maia1\data\politiekh\masters\nlp\project_lab1


## 3) Load data from Lab1

In [3]:
raw_path = LAB1_ROOT / 'data' / 'raw.jsonl'
labels_path = LAB1_ROOT / 'data' / 'labels.csv'

rows = []
with raw_path.open('r', encoding='utf-8') as f:
    for line in f:
        obj = json.loads(line)
        text = obj.get('text') or obj.get('content') or obj.get('review') or obj.get('comment')
        text_id = obj.get('text_id') or obj.get('id')
        rows.append({'text_id': text_id, 'raw_text': text})

df_raw = pd.DataFrame(rows).dropna(subset=['raw_text']).copy()
df_labels = pd.read_csv(labels_path)
df_in = df_raw.merge(df_labels, on='text_id', how='inner')
print('Loaded:', df_in.shape)
df_in.head()

Loaded: (1000, 3)


,text_id,raw_text,label
0,9905,"Вступив на ІСТ цього року, тепер молюся, щоб п...",Question / Request for Help
1,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,Question / Request for Help
2,3099,Старий університет поки що вчить. Наразі налаш...,Neutral Comment
3,8664,"На пл. Ринок ЦНАП м.Львова, швидке ьа якісне в...",Gratitude / Positive Feedback
4,1035,"Мені здається, що наша кузня супер-кадрів в IT...",Suggestion / Idea


## 4) Run preprocessing → processed_v2

In [4]:
out_dir = LAB2_ROOT / 'data' / 'processed_v2'
out_dir.mkdir(parents=True, exist_ok=True)

processed_rows = []
stats_sum = {'repl_url':0, 'repl_email':0, 'repl_phone':0}

for r in tqdm(df_in.to_dict(orient='records'), total=len(df_in)):
    res = preprocess(r['raw_text'])
    processed_rows.append({
        'text_id': r['text_id'],
        'text': res.masked,
        'sentences': json.dumps(res.sentences, ensure_ascii=False),
        'label': r['label'],
    })
    for k in ['repl_url','repl_email','repl_phone']:
        stats_sum[k] += res.stats[k]

df_v2 = pd.DataFrame(processed_rows)
df_v2.to_csv(out_dir / 'processed_v2.csv', index=False, encoding='utf-8')
print('Saved:', out_dir / 'processed_v2.csv')
print('Mask counts:', stats_sum)
df_v2.head()

  0%|          | 0/1000 [00:00<?, ?it/s]

 55%|█████▌    | 553/1000 [00:00<00:00, 5504.43it/s]

100%|██████████| 1000/1000 [00:00<00:00, 5519.64it/s]

Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab2\data\processed_v2\processed_v2.csv
Mask counts: {'repl_url': 12, 'repl_email': 1, 'repl_phone': 11}


,text_id,text,sentences,label
0,9905,"Вступив на ІСТ цього року, тепер молюся, щоб п...","[""Вступив на ІСТ цього року, тепер молюся, щоб...",Question / Request for Help
1,10001201,Цифрова держава Повідомлення 123 від 18.04.202...,"[""Цифрова держава Повідомлення 123 від 18.04.2...",Question / Request for Help
2,3099,Старий університет поки що вчить. Наразі налаш...,"[""Старий університет поки що вчить."", ""Наразі ...",Neutral Comment
3,8664,"На пл. Ринок ЦНАП м.Львова, швидке ьа якісне в...","[""На пл. Ринок ЦНАП м.Львова, швидке ьа якісне...",Gratitude / Positive Feedback
4,1035,"Мені здається, що наша кузня супер-кадрів в IT...","[""Мені здається, що наша кузня супер-кадрів в ...",Suggestion / Idea


## 5) Show 15 raw → processed_v2 examples

In [5]:
show = df_in.sample(15, random_state=42).copy()
show['after'] = show['raw_text'].apply(lambda x: preprocess(x).masked)
show[['text_id','label','raw_text','after']]

,text_id,label,raw_text,after
521,7043,Complaint / Dissatisfaction,Операторів 12. А фотографів 6. З них пів дня п...,Операторів 12. А фотографів 6. З них пів дня п...
737,9294,Neutral Comment,"Чернігівська область, м. Мена, вул. Сіверський...","Чернігівська область, м. Мена, вул. Сіверський..."
740,3825,Complaint / Dissatisfaction,"Напевно, був би кращім місцем, якби не було ко...","Напевно, був би кращім місцем, якби не було ко..."
660,925,Neutral Comment,"Щоб там щодня, крім вихідних. Нічого особливог...","Щоб там щодня, крім вихідних. Нічого особливог..."
411,10001377,Question / Request for Help,Тільки в епіцентрі працює? А в леруа мерлен на...,Тільки в епіцентрі працює? А в леруа мерлен на...
678,1600,Neutral Comment,Реєстрація ФОП покроково з врахуванням нововве...,Реєстрація ФОП покроково з врахуванням нововве...
626,16151,Neutral Comment,Навчальний заклад розташований в найголовнішій...,Навчальний заклад розташований в найголовнішій...
513,1871,Suggestion / Idea,Це незвичайний навчальний заклад. Якщо шукаєте...,Це незвичайний навчальний заклад. Якщо шукаєте...
859,4358,Neutral Comment,Харківський університет називають каразінським...,Харківський університет називають каразінським...
136,16189,Gratitude / Positive Feedback,"Як в живу, як на фото, дуже гарно виглядає, сю...","Як в живу, як на фото, дуже гарно виглядає, сю..."


## 6) Before/After audit stats + quality checks

In [6]:
def lens_stats(s: pd.Series):
    char_len = s.fillna('').astype(str).str.len()
    word_len = s.fillna('').astype(str).apply(lambda x: len(str(x).split()))
    return {
        'char_mean': float(char_len.mean()),
        'char_median': float(char_len.median()),
        'word_mean': float(word_len.mean()),
        'word_median': float(word_len.median()),
        'short_lt5_pct': float((word_len < 5).mean() * 100),
        'empty_pct': float((s.fillna('').astype(str).str.strip() == '').mean() * 100),
        'dup_pct': float(s.fillna('').astype(str).duplicated(keep=False).mean() * 100),
    }

before = lens_stats(df_in['raw_text'])
after = lens_stats(df_v2['text'])
before, after

({'char_mean': 152.615,
  'char_median': 124.5,
  'word_mean': 22.509,
  'word_median': 19.0,
  'short_lt5_pct': 0.2,
  'empty_pct': 0.0,
  'dup_pct': 0.4},
 {'char_mean': 151.803,
  'char_median': 124.0,
  'word_mean': 22.503,
  'word_median': 19.0,
  'short_lt5_pct': 0.2,
  'empty_pct': 0.0,
  'dup_pct': 0.4})

## 7) Edge cases run + show examples

In [7]:
edge_path = LAB2_ROOT / 'tests' / 'edge_cases.jsonl'
edge = []
with edge_path.open('r', encoding='utf-8') as f:
    for line in f:
        edge.append(json.loads(line))
df_edge = pd.DataFrame(edge)
df_edge['after'] = df_edge['raw_text'].apply(lambda x: preprocess(x).masked)
df_edge[['id','expected_behavior','raw_text','after']].head(12)

,id,expected_behavior,raw_text,after
0,ec01,не розбивати речення після 'м.'; тире нормаліз...,м. Львів — гарне місто. Але є черги.,м. Львів - гарне місто. Але є черги.
1,ec02,не розбивати після 'вул.'; лишити час; sentenc...,"вул. Шевченка, 10. Працює до 18:00.","вул. Шевченка, 10. Працює до 18:00."
2,ec03,не розбивати всередині 3.14; 'грн.' не має лам...,Ціна 3.14 грн. Це тест.,Ціна 3.14 грн. Це тест.
3,ec04,не розбивати 1.2.3; split по '!'; output 2 реч...,Версія 1.2.3 не працює! Допоможіть.,Версія 1.2.3 не працює! Допоможіть.
4,ec05,email -> <EMAIL>.,Напишіть на test@example.com будь ласка.,Напишіть на <EMAIL> будь ласка.
5,ec06,phone -> <PHONE>.,Мій номер +38 (099) 123-45-67,Мій номер <PHONE>
6,ec07,url -> <URL>.,Деталі на https://site.com/page?id=1,Деталі на <URL>
7,ec08,латинська i в кириличному слові -> 'Львів'.,Львiв дуже красивий.,Львів дуже красивий.
8,ec09,"лапки -> """"""; тире -> -; split коректний.",«Супер сервіс!» — дякую.,"""Супер сервіс!"" - дякую."
9,ec10,зберегти емоційність; split на 2 речення.,Жахливо... Нікому не раджу!!!,Жахливо... Нікому не раджу!!!


## 8) Mini-regression: idempotence + no empty explosions

In [8]:
idem_fail = []
empty_fail = []
for r in edge:
    x = r['raw_text']
    if not idempotence_check(x):
        idem_fail.append(r['id'])
    if not no_empty_explosion(x):
        empty_fail.append(r['id'])
print('Idempotence fails:', idem_fail)
print('Empty explosion fails:', empty_fail)

Idempotence fails: []
Empty explosion fails: []


## 9) Generate docs/audit_summary_lab2.md

In [9]:
doc_path = LAB2_ROOT / 'docs' / 'audit_summary_lab2.md'
lines = []
lines.append('# Audit summary — Lab2\n')
lines.append('## Before vs After\n')
lines.append('### Before (raw_text)')
for k,v in before.items():
    lines.append(f'- {k}: {v:.4f}')
lines.append('\n### After (processed_v2)')
for k,v in after.items():
    lines.append(f'- {k}: {v:.4f}')
lines.append('\n## Masking counts (total)')
for k,v in stats_sum.items():
    lines.append(f'- {k}: {v}')
doc_path.write_text('\n'.join(lines), encoding='utf-8')
print('Saved:', doc_path)

Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab2\docs\audit_summary_lab2.md


## 10) Auto-insert examples into preprocess_policy.md

In [10]:
policy_md = LAB2_ROOT / 'docs' / 'preprocess_policy.md'
policy_examples = df_in.sample(12, random_state=7).copy()
policy_examples['after'] = policy_examples['raw_text'].apply(lambda x: preprocess(x).masked)

section_title = "## Авто-приклади 'до/після' (з реальних даних)"
out = [section_title, '']
for _, row in policy_examples.iterrows():
    out.append(f"- **до:** {row['raw_text']}")
    out.append(f"  **після:** {row['after']}")
    out.append('')

existing = policy_md.read_text(encoding='utf-8')
if section_title in existing:
    existing = existing.split(section_title)[0].rstrip()

new_section = '\n'.join(out).rstrip() + '\n'
policy_md.write_text(existing + '\n\n' + new_section, encoding='utf-8')
print('Updated:', policy_md)


Updated: C:\Users\maia1\data\politiekh\masters\nlp\project_lab2\docs\preprocess_policy.md


## 11) Create a small sample for GitHub (optional)

In [11]:
sample_dir = LAB2_ROOT / 'data' / 'sample'
sample_dir.mkdir(exist_ok=True)
df_v2.sample(50, random_state=42).to_csv(sample_dir / 'processed_v2_sample.csv', index=False, encoding='utf-8')
print('Saved sample:', sample_dir / 'processed_v2_sample.csv')

Saved sample: C:\Users\maia1\data\politiekh\masters\nlp\project_lab2\data\sample\processed_v2_sample.csv


In [12]:
card_path = LAB2_ROOT / 'docs' / 'dataset_card.md'

lines = []
lines.append('# Dataset Card — Lab2 update')
lines.append('')
lines.append('## Source & task')
lines.append('- Base dataset: UAReviews subset from Lab1 (N=1000, 5 balanced classes).')
lines.append('- Task: Track A text classification.')
lines.append('')
lines.append('## Preprocessing (Lab2)')
lines.append('- whitespace/NBSP normalization; apostrophes/quotes/dashes canonicalization')
lines.append('- masking: <URL>, <EMAIL>, <PHONE>')
lines.append('- robust sentence split with UA abbreviation and decimal protection')
lines.append('')
lines.append('## Before vs After snapshot')
lines.append(f"- char_mean: {before['char_mean']:.3f} -> {after['char_mean']:.3f}")
lines.append(f"- word_mean: {before['word_mean']:.3f} -> {after['word_mean']:.3f}")
lines.append(f"- short_lt5_pct: {before['short_lt5_pct']:.3f}% -> {after['short_lt5_pct']:.3f}%")
lines.append(f"- dup_pct: {before['dup_pct']:.3f}% -> {after['dup_pct']:.3f}%")
lines.append('')
lines.append('## Masking totals')
lines.append(f"- URL replacements: {stats_sum['repl_url']}")
lines.append(f"- EMAIL replacements: {stats_sum['repl_email']}")
lines.append(f"- PHONE replacements: {stats_sum['repl_phone']}")
lines.append('')
lines.append('## Risks')
lines.append('- semantic noise and slang remain in text (by design)')
lines.append('- near-duplicates and domain shift are still possible')

card_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print('Saved:', card_path)


Saved: C:\Users\maia1\data\politiekh\masters\nlp\project_lab2\docs\dataset_card.md
